In [ ]:
import requests

url = "https://www.nab.com.au"

# Some sites block requests that don't look like a real browser, so we send a browser-like User-Agent header
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36"
}

response = requests.get(url, headers=headers, timeout=15)

print("Status code:", response.status_code)
print("Content length (characters):", len(response.text))
print("---- first 2000 characters of what came back ----")
print(response.text[:2000])

In [ ]:
from bs4 import BeautifulSoup

soup = BeautifulSoup(response.text, "html.parser")

# Pull out the key structured signals first — these are gold for grounding
title = soup.title.string if soup.title else "(no title)"
meta_desc_tag = soup.find("meta", attrs={"name": "description"})
meta_desc = meta_desc_tag["content"] if meta_desc_tag else "(no meta description)"

# Then the readable body text, with scripts and styles removed
for tag in soup(["script", "style", "noscript"]):
    tag.decompose()   # strip these out entirely — they're not readable content

body_text = soup.get_text(separator=" ", strip=True)

print("TITLE:", title)
print("META DESCRIPTION:", meta_desc)
print("---- readable text length:", len(body_text), "characters ----")
print(body_text[:2000])

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
api_key = os.getenv("OPENROUTER_API_KEY")

print("Key loaded:", api_key is not None)
print("Key starts with:", api_key[:7] if api_key else "NOT FOUND")

In [ ]:
import requests
import json

# --- Config: which audit mode ---
AUDIENCE = "Personal"   # B2C. For B2B you'd use "Business" or "Corporate".
BRAND_NAME = "NAB"

# body_text is the parsed NAB homepage text from the earlier cell

system_prompt = f"""You are an AEO (Answer Engine Optimization) analyst. Your job is to generate the set of questions that real {AUDIENCE.lower()} customers would ask an AI assistant (like ChatGPT) when researching products in {BRAND_NAME}'s category — WITHOUT naming {BRAND_NAME}. These are the prompts we will later run against AI models to measure whether {BRAND_NAME} gets mentioned.

Ground your analysis ONLY in the provided website content. Identify the {AUDIENCE}-facing product families, then for each one generate customer-style questions across four funnel stages:
- problem_aware: the customer feels a problem but doesn't know solutions yet
- solution_aware: the customer knows the solution category and is exploring options
- comparison: the customer is comparing providers or products
- bottom_funnel: the customer is close to choosing, asking specific/practical questions

Rules:
- Questions must be natural, the way a real person types into an AI. Not keyword-stuffed.
- Do NOT mention {BRAND_NAME} in any question. These measure unbranded visibility.
- Include location context (Australia) where a real customer would.
- Return ONLY valid JSON, no markdown fences, no text outside the JSON.

Return this exact structure:
{{
  "product_families": [
    {{
      "family": "the product family name",
      "prompts": [
        {{"funnel_stage": "problem_aware", "question": "..."}},
        {{"funnel_stage": "solution_aware", "question": "..."}},
        {{"funnel_stage": "comparison", "question": "..."}},
        {{"funnel_stage": "bottom_funnel", "question": "..."}}
      ]
    }}
  ]
}}"""

user_prompt = f"Website content for {BRAND_NAME} ({AUDIENCE} focus):\n\n{body_text}"

response = requests.post(
    "https://openrouter.ai/api/v1/chat/completions",
    headers={"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"},
    json={
        "model": "~deepseek/deepseek-v4-flash-latest",
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        "temperature": 0.7,
        "response_format": {"type": "json_object"}
    },
    timeout=90
)

print("Status:", response.status_code)
data = response.json()
raw = data["choices"][0]["message"]["content"]
print("---- raw model output (first 1500 chars) ----")
print(raw[:1500])

In [ ]:
# Defensive parse — same lesson as the LinkedIn tool: models sometimes add
# stray whitespace or fences, so don't trust raw JSON blindly.
import pandas as pd
def parse_json_safely(text):
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        # fallback: grab everything between the first { and last }
        start = text.find("{")
        end = text.rfind("}")
        if start != -1 and end != -1:
            return json.loads(text[start:end+1])
        raise

parsed = parse_json_safely(raw)

# Flatten the nested structure into rows: one row per prompt.
# Each prompt belongs to a family and a funnel stage — that's a flat table.
rows = []
for family_obj in parsed["product_families"]:
    family = family_obj["family"]
    for prompt_obj in family_obj["prompts"]:
        rows.append({
            "product_family": family,
            "funnel_stage": prompt_obj["funnel_stage"],
            "question": prompt_obj["question"]
        })

prompts_df = pd.DataFrame(rows)

print("Total prompts generated:", prompts_df.shape[0])
print("Product families covered:", prompts_df["product_family"].nunique())

In [ ]:
import time

# --- The prompts to test: just the Credit cards family ---
credit_card_prompts = prompts_df[prompts_df["product_family"] == "Credit cards"]

# --- The models to test ---
models_to_test = [
    "openai/gpt-5.6-luna:online",
    "anthropic/claude-sonnet-5:online",
    "anthropic/claude-opus-5:online",
    "google/gemini-3.7-flash:online",
    "perplexity/sonar",              # native search, no :online
]

BRAND = "NAB"

def run_one(model, question):
    """Fire one prompt at one model with search on. Returns a result dict."""
    try:
        resp = requests.post(
            "https://openrouter.ai/api/v1/chat/completions",
            headers={"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"},
            json={
                "model": model,
                "messages": [{"role": "user", "content": question}],
            },
            timeout=120
        )
        resp.raise_for_status()
        data = resp.json()
        msg = data["choices"][0]["message"]
        answer = msg.get("content", "") or ""

        # Extract citations from the url_citation annotations
        citations = []
        for ann in msg.get("annotations", []) or []:
            if ann.get("type") == "url_citation":
                url = ann.get("url_citation", {}).get("url")
                if url:
                    citations.append(url)

        return {
            "model_requested": model,
            "model_actual": data.get("model", ""),   # what OpenRouter actually used
            "brand_mentioned": BRAND.lower() in answer.lower(),
            "answer": answer,
            "citations": citations,
            "num_citations": len(citations),
            "error": None,
        }
    except Exception as e:
        return {
            "model_requested": model,
            "model_actual": "",
            "brand_mentioned": None,
            "answer": "",
            "citations": [],
            "num_citations": 0,
            "error": f"{type(e).__name__}: {e}",
        }

# --- Run the full grid, printing progress so you can watch it ---
results = []
for _, prow in credit_card_prompts.iterrows():
    q = prow["question"]
    for model in models_to_test:
        print(f"  {model:45s} | {prow['funnel_stage']:15s} ...", end="", flush=True)
        r = run_one(model, q)
        r["funnel_stage"] = prow["funnel_stage"]
        r["question"] = q
        r["product_family"] = prow["product_family"]
        results.append(r)
        status = r["error"] if r["error"] else f"mentioned={r['brand_mentioned']} cites={r['num_citations']}"
        print(f" {status}")
        time.sleep(1)   # gentle pacing so we don't hammer the API

results_df = pd.DataFrame(results)
print("\nDone. Total calls:", len(results_df))

In [ ]:
import json

# Full-structure save (citations preserved as real lists)
results_df.to_json("nab_credit_cards_results.json", orient="records", indent=2)

# Flat CSV for eyeballing (citations become a string here — that's fine, JSON is the real save)
results_df.to_csv("nab_credit_cards_results.csv", index=False)

print("Saved nab_credit_cards_results.json and .csv")

In [ ]:
results_df = pd.read_json("nab_credit_cards_results.json", orient="records")

In [ ]:
prompts_df.to_json("nab_prompts.json", orient="records", indent=2)
prompts_df.to_csv("nab_prompts.csv", index=False)
print("Saved nab_prompts.json and nab_prompts.csv —", prompts_df.shape[0], "prompts")